In [1]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np
from functools import partial
import joblib

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

/opt/conda/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

Define path variables

In [4]:
representation_type = "traditional_word_BOW"
developer_initials = "JP"

In [5]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "traditional"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"


Connect to databricks for logging results

In [6]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/14 21:27:50 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/14 21:27:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/14 21:27:50 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2025/12/14 21:27:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


What are GPU are the experiments run on

In [7]:
!nvidia-smi

Sun Dec 14 21:27:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:A3:00.0 Off |                    0 |
|  0%   35C    P8             30W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
running_on_gpu = torch.cuda.is_available()

In [9]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [10]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

89

In [11]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Classification threshold constant specification

In [12]:
classification_thresholds = [x/1000 for x in range(200, 999)]

TF-IDF parameters for tuning

In [13]:
maximum_features = [20_000, 50_000, 100_000, None]
lowercase = [False, True]
minimum_df = (1e-5, 0.01)
maximum_df = (0.7, 1.0)
binary = [False, True]

# Load dataset

#### Load training data

In [14]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [00:24<00:00, 458MB/s]


Successfully loaded 273301 items.


In [15]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [16]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Load validation data

In [17]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:00<00:00, 438MB/s] 


Successfully loaded 2500 items.


In [18]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [19]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:01<00:00, 494MB/s] 


Successfully loaded 19999 items.


In [20]:
test_data_df = pd.DataFrame(test_data)

# Extract features functions

In [21]:
def build_bow_vectorizer(max_features, min_df, max_df, lowercase=True, binary=False):
    return CountVectorizer(
        analyzer="word",
        ngram_range=(1, 1),
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
        lowercase=lowercase,
        binary=binary
    )


# Evaluation functions

Evaluation function

In [22]:
def evaluate_results(y_true, y_pred, average='binary'):
    accuracy = round(accuracy_score(y_true, y_pred)*100, 2)
    precision = round(precision_score(y_true, y_pred, average=average, zero_division=0)*100, 2)
    recall = round(recall_score(y_true, y_pred, average=average)*100, 2)
    f1 = round(f1_score(y_true, y_pred, average=average)*100, 2)
    return accuracy, precision, recall, f1

Create the confusion matrix

In [23]:
def plot_confusion_matrix(y_true, y_pred, model_id="", labels=[False, True]):

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
        cbar=False
    )

    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix: {model_id}')

    plt.tight_layout()
    return fig

Create histogram of F1 score for different thresholds

In [24]:
def plot_f1_vs_threshold(results_df):
    plt.figure(figsize=(9, 5))
    plt.plot(results_df["threshold"], results_df["f1"], linewidth=2)
    plt.xlabel("Threshold")
    plt.ylabel("F1 Score")
    plt.title("F1 Score vs Classification Threshold")
    plt.grid(True)
    plt.tight_layout()
    return plt.gcf()

# Cosine similarity of embeddings functions

Evaluate model function

In [25]:
def compute_similarities(train_df, test_df, vectorizer):
    start_time = time.time()
    verification_results = []

    def extract_all_texts(train_df):
        texts = []
        for pair in tqdm(train_df["pair"], desc="Extracting texts"):
            texts.extend(pair)
        return texts
    
    all_texts = extract_all_texts(train_df)
    print("Fitting texts to the vectorizer")
    vectorizer.fit(all_texts)
    
    for i in tqdm(test_data_df.index, desc="Processing rows"):
        resulting_df_row = {}
        resulting_df_row['id']  = test_data_df.loc[i, 'id']
        resulting_df_row['actual_result'] = test_data_df.loc[i, 'same']
        text1 = test_data_df.loc[i, 'pair'][0]
        text2 = test_data_df.loc[i, 'pair'][1]
        
        text1_embedding = vectorizer.transform([text1])
        text2_embedding = vectorizer.transform([text2])

        cosine_similarity_score = cosine_similarity(
            text1_embedding,
            text2_embedding
        )[0, 0]

        resulting_df_row['cosine_similarity'] = cosine_similarity_score
        verification_results.append(resulting_df_row)

    result_df = pd.DataFrame(verification_results)
    print("--- Execution Time: %s seconds ---" % round(time.time() - start_time, 2))

    return result_df

Evaluate classification thresholds

In [26]:
def evaluate_classification_thresholds(result_df, classification_thresholds):
    results = []
    y_embeddings = result_df['cosine_similarity']
    y_true = result_df['actual_result']
    for threshold in classification_thresholds:
        y_pred = (result_df["cosine_similarity"] >= threshold).astype(int)
        accuracy, precision, recall, f1 = evaluate_results(y_true, y_pred)
        results.append({
            "threshold": threshold,
            "accuracy" : accuracy,
            "precision" : precision,
            "recall" : recall,
            "f1" : f1
        })
    return pd.DataFrame(results)

# Optuna study function - for cosine similarity of embeddings

In [27]:
def objective(trial):
    top_f1_trial = -1


    suggested_max_features = trial.suggest_categorical("max_features", maximum_features)
    suggested_min_df = trial.suggest_float("min_df", minimum_df[0], minimum_df[1])
    suggested_max_df =  trial.suggest_float("max_df", maximum_df[0], maximum_df[1])
    suggested_lowercase = trial.suggest_categorical("lowercase", lowercase)
    suggested_binary = trial.suggest_categorical("binary", binary)
    
    print(f"Starting the optuna trial {trial.number} \n")

    vectorizer = build_bow_vectorizer(
        max_features=suggested_max_features,
        min_df=suggested_min_df, 
        max_df=suggested_max_df,
        lowercase=suggested_lowercase,
        binary=suggested_binary)

    print(f"Computing similarities\n")
    result_df = compute_similarities(train_data_df, val_data_df, vectorizer)

    print(f"Evaluating classification thresholds \n")
    threshold_results_df = evaluate_classification_thresholds(result_df, classification_thresholds)

    top_row = threshold_results_df.sort_values("f1", ascending=False).iloc[0]
    top_threshold = top_row["threshold"]
    top_f1 = top_row["f1"]
    top_accuracy = top_row["accuracy"]
    top_precision = top_row["precision"]
    top_recall = top_row["recall"]

    print(f"Logging results \n")
    print(f"Top threshold: {top_threshold}\n")
    print(f"Top f1: {top_f1}\n")
    print(f"Top accuracy: { top_accuracy}\n")
    print(f"Top precision: {top_precision}\n")
    print(f"Top recall: {top_recall}\n")

    print(f"Used hyperparameters \n")
    print(f"max_features: {suggested_max_features}\n")
    print(f"min_df: {suggested_min_df}\n")
    print(f"max_df: {suggested_max_df}\n")
    print(f"lowercase: {suggested_lowercase}\n")
    print(f"binary: {suggested_binary}\n")

    trial.set_user_attr(
    "results",
    {
        "metrics": {
            "top_threshold": top_threshold,
            "top_f1": top_f1,
            "top_accuracy": top_accuracy,
            "top_precision": top_precision,
            "top_recall": top_recall,
        },
        "hyperparameters": {
            "max_features": suggested_max_features,
            "min_df": suggested_min_df,
            "max_df": suggested_max_df,
            "lowercase": suggested_lowercase,
            "binary": suggested_binary
        },
    }
    )

    return top_f1   

# Evaluate cosine embeddings

cosine embedding constants

In [28]:
classifier = "embedding_cosine_similarity"

Set up the study

In [29]:
date_str = datetime.now().strftime("%Y%m%d_%H%M")

In [30]:
optuna_study_path = f"sqlite:///optuna_study_{developer_initials}_{representation_type}_{classifier}"

study = optuna.create_study(
    study_name=f"authorship_verification_{representation_type}_{classifier}",
    storage=optuna_study_path,
    direction="maximize",
    load_if_exists=True
)

[I 2025-12-14 21:28:19,479] A new study created in RDB with name: authorship_verification_traditional_word_BOW_embedding_cosine_similarity


In [ ]:
study.optimize(objective, n_trials=30)

Starting the optuna trial 0 

Computing similarities



Extracting texts: 100%|██████████| 273301/273301 [00:00<00:00, 669802.96it/s] 


Fitting texts to the vectorizer


Processing rows: 100%|██████████| 19999/19999 [01:27<00:00, 227.74it/s]


--- Execution Time: 863.15 seconds ---
Evaluating classification thresholds 

Logging results 

Top threshold: 0.271

Top f1: 73.52

Top accuracy: 70.5

Top precision: 66.7

Top recall: 81.91

Used hyperparameters 

max_features: 100000

min_df: 0.0037469367904146883

max_df: 0.8012961975447327

lowercase: True

binary: True



...

[I 2025-12-14 21:56:19,969] Trial 1 finished with value: 63.44 and parameters: {'max_features': None, 'min_df': 0.004095799563155718, 'max_df': 0.7797290008687262, 'lowercase': True, 'binary': False}. Best is trial 0 with value: 73.52.


Starting the optuna trial 2 

Computing similarities



Extracting texts: 100%|██████████| 273301/273301 [00:00<00:00, 451443.99it/s]


Fitting texts to the vectorizer


Processing rows:  34%|███▎      | 6700/19999 [00:28<00:55, 240.56it/s]

Load the trial data

In [ ]:
trial_rows = []

for trial in study.trials:
    
    metrics = trial.user_attrs["results"]["metrics"]
    hyper_params = trial.user_attrs["results"]["hyperparameters"]

    row = {
        "trial_number": trial.number,
        "top_threshold": metrics["top_threshold"],
        "top_f1": metrics["top_f1"],
        "top_accuracy": metrics["top_accuracy"],
        "top_precision": metrics["top_precision"],
        "top_recall": metrics["top_recall"],
        "max_features" : hyper_params["max_features"],
        "min_df" : hyper_params["min_df"],
        "max_df" : hyper_params["max_df"],
        "lowercase" : hyper_params["lowercase"],
        "binary" : hyper_params["binary"],
    }
    trial_rows.append(row)

trial_results_df = pd.DataFrame(trial_rows)

In [ ]:
trial_results_df.head()

Load the data from the best trial to variables

In [ ]:
best_trial = study.best_trial
best_trial_data = best_trial.user_attrs["results"]

In [ ]:
top_threshold = best_trial_data["metrics"]["top_threshold"]
top_max_features = best_trial_data["hyperparameters"]["max_features"]
top_min_df = best_trial_data["hyperparameters"]["min_df"]
top_max_df = best_trial_data["hyperparameters"]["max_df"]
top_lowercase = best_trial_data["hyperparameters"]["lowercase"]
top_binary = best_trial_data["hyperparameters"]["binary"]

# Final testing

In [ ]:
best_model_path = current_dir.parent.parent / "data" / "02_models" / f"{developer_initials}_{representation_type}_{classifier}_best_model.pkl"

Reconstruct the best vectorizer

In [ ]:
best_vectorizer = build_bow_vectorizer(
    max_features=top_max_features,
    min_df=top_min_df,
    max_df=top_max_df,
    lowercase=top_lowercase,
    binary=top_binary
    )

Combine validation and training dataset

In [ ]:
final_train_df = pd.concat(
    [train_data_df, val_data_df],
    ignore_index=True
)

In [ ]:
result_df = compute_similarities(final_train_df, test_data_df, best_vectorizer)
threshold_results_df = evaluate_classification_thresholds(result_df, classification_thresholds)

In [ ]:
y_scores = result_df["cosine_similarity"].values

In [ ]:
y_pred = (y_scores >= top_threshold).astype(int)
y_true = result_df["actual_result"].astype(int).values

Evaluate the final results

In [ ]:
top_accuracy, top_precision, top_recall, top_f1 =  evaluate_results(y_true, y_pred, average='binary')

In [ ]:
print(f"Logging results \n")
print(f"Top threshold: {top_threshold}\n")
print(f"Top f1: {top_f1}\n")
print(f"Top accuracy: { top_accuracy}\n")
print(f"Top precision: {top_precision}\n")
print(f"Top recall: {top_recall}\n")

print(f"Used hyperparameters \n")
print(f"max_features: {top_max_features}\n")
print(f"min_df: {top_min_df}\n")
print(f"max_df: {top_max_df}\n")
print(f"lowercase: {top_lowercase}\n")
print(f"binary: {top_binary}\n")

Dump the model

In [ ]:
joblib.dump(best_vectorizer, best_model_path)

In [ ]:
f1_threshold_histogram_fig = plot_f1_vs_threshold(threshold_results_df)
f1_threshold_histogram_fig.show()

Confusion matrix

In [ ]:
cm_fig = plot_confusion_matrix(y_true, y_pred, f"{representation_type}_{classifier}", labels=[False, True])

Log model information + metrics + results table + confusion matrix

In [ ]:
mlflow.set_experiment("/Users/jiripokorny455@gmail.com/JP_authorship_verification_dt")
with mlflow.start_run(run_name=f"{developer_initials}_{representation_type}_{classifier}"):
    mlflow.log_param("gpu_name", gpu_name)
    mlflow.log_param("gpu_vram_gb", gpu_vram_gb)
    
    mlflow.log_param("representation_type", representation_type )
    mlflow.log_param("classifier", classifier)
    mlflow.log_table(data=trial_results_df, artifact_file="trial_results.json")
    
    mlflow.log_param("max_features", top_max_features)
    mlflow.log_param("min_df", top_min_df)
    mlflow.log_param("max_df", top_min_df)
    mlflow.log_param("lowercase", top_lowercase)
    mlflow.log_param("binary", binary)

    mlflow.log_param("classification_score_thresholds", classification_thresholds)
    mlflow.log_table(data=result_df, artifact_file="embedding_similarity_results-test_data.json")

    mlflow.log_metric("top_threshold", top_threshold)
    mlflow.log_metric("top_accuracy", top_accuracy)
    mlflow.log_metric("top_precision", top_precision)
    mlflow.log_metric("top_recall", top_recall)
    mlflow.log_metric("top_f1", top_f1)

    mlflow.log_figure(f1_threshold_histogram_fig, "f1_results_threshold_histogram-test_data.png")
    mlflow.log_figure(cm_fig, "confusion_matrix-test_data.png")